# 07 · Rango sugerido de precios (regresión cuantílica)

El tablero muestra, además del precio de referencia, un **rango sugerido**. Se obtiene entrenando dos modelos LightGBM con pérdida cuantílica: uno estima el percentil inferior del precio y otro el superior para un alojamiento con esas características.

Se evalúa con septiembre, solo en las localidades habilitadas, y se registran:

| Métrica | Qué mide | Meta |
|---|---|---|
| `cobertura_nuevos` | % de precios reales de anuncios nuevos dentro del rango | ≈ 80 % |
| `cobertura` | Lo mismo sobre toda la captura de septiembre | ≈ 80 % |
| `ancho_mediano_cop` | Ancho típico del rango en pesos | Lo más angosto posible |
| `razon_sup_inf` | Cuántas veces el límite superior es el inferior | Lo más bajo posible |
| `desvio_meta` | Distancia entre `cobertura_nuevos` y 80 % | 0 |

Un buen rango cubre cerca del 80 % de los precios **sin ser tan ancho que deje de ser útil** para el anfitrión. Las corridas van al experimento `bogoprice-intervalos` de MLflow.

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd

from bogoprice import variables_v2
from bogoprice.experimentos import preparar_datos
from bogoprice.metrics import cobertura
from bogoprice.tracking import iniciar_mlflow, registrar_corrida

# Tag de autor de MLflow
AUTOR = "Juan Camilo Piñeros Guzmán"
iniciar_mlflow("bogoprice-intervalos")
datos = preparar_datos()
resultados = []


def intervalo(nombre, version, a_inf, a_sup, registrar=True, **p):
    """Entrena los modelos del percentil inferior y superior, evalúa en septiembre
    y registra la corrida en MLflow. Devuelve las métricas."""
    # Valores base de los dos modelos cuantílicos
    base = {"n_estimators": 1500, "learning_rate": 0.1, "num_leaves": 63,
            "min_child_samples": 30, "subsample": 0.8, "subsample_freq": 1,
            "colsample_bytree": 0.7}
    params = base | p
    cols = variables_v2.columnas(version)

    def modelo(alpha):
        return variables_v2.construir_pipeline(
            lgb.LGBMRegressor(objective="quantile", alpha=alpha, verbose=-1, random_state=0, **params),
            version)

    q_inf = modelo(a_inf).fit(datos.train[cols], datos.y_train)
    q_sup = modelo(a_sup).fit(datos.train[cols], datos.y_train)
    lim_inf = np.exp(q_inf.predict(datos.test[cols]))
    lim_sup = np.exp(q_sup.predict(datos.test[cols]))
    y, nuevo = datos.y_test_cop, datos.es_nuevo

    m = {
        "cobertura": cobertura(y, lim_inf, lim_sup),
        "cobertura_nuevos": cobertura(y[nuevo], lim_inf[nuevo], lim_sup[nuevo]),
        "ancho_mediano_cop": float(np.median(lim_sup - lim_inf)),
        "razon_sup_inf": float(np.median(lim_sup / lim_inf)),
    }
    m["desvio_meta"] = abs(m["cobertura_nuevos"] - 80)
    if registrar:
        registrar_corrida(nombre,
                          {"modelo": "LightGBM cuantílico", "conjunto": version,
                           "alpha_inf": a_inf, "alpha_sup": a_sup, **params,
                           "parametro_variado": p and next(iter(p)) or "percentiles"},
                          m, tags={"autor": AUTOR, "conjunto": version, "modelo": "Intervalo"})
    return m

2026/11/08 18:56:26 INFO mlflow.tracking.fluent: Experiment with name 'bogoprice-intervalos' does not exist. Creating a new experiment.


P99 junio: $1,330,087 | train 17749 | test 17561 | nuevos 1632 | localidades 9 | tipologías 17


## 1. Percentiles del rango

Con los hiperparámetros base, se prueban **4 pares de percentiles**. Mientras más separados, más cobertura pero más ancho el rango.

In [2]:
percentiles = [(0.10, 0.90), (0.075, 0.925), (0.05, 0.95), (0.025, 0.975)]

for version in ["v2", "v2_geo"]:
    for a_inf, a_sup in percentiles:
        etiqueta = f"P{a_inf * 100:g}-P{a_sup * 100:g}"
        nombre = f"intervalo_{etiqueta}_{version}"
        m = intervalo(nombre, version, a_inf, a_sup)
        resultados.append({"corrida": nombre, "conjunto": version, "parametro": "percentiles",
                           "valor": etiqueta, **m})
        print(f"{version:7s} {etiqueta:12s} cobertura nuevos = {m['cobertura_nuevos']:.1f} %  "
              f"ancho = ${m['ancho_mediano_cop']:,.0f}")

🏃 View run intervalo_P10-P90_v2 at: http://localhost:8050/#/experiments/2/runs/50eee1b503974a6b9c30cc099139f1c8
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2      P10-P90      cobertura nuevos = 77.3 %  ancho = $121,063


🏃 View run intervalo_P7.5-P92.5_v2 at: http://localhost:8050/#/experiments/2/runs/0dee344888b548a4aee8383effc9e74c
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2      P7.5-P92.5   cobertura nuevos = 81.5 %  ancho = $139,983


🏃 View run intervalo_P5-P95_v2 at: http://localhost:8050/#/experiments/2/runs/6b056f0712fe479cbe4807191caa3bed
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2      P5-P95       cobertura nuevos = 87.4 %  ancho = $166,382


🏃 View run intervalo_P2.5-P97.5_v2 at: http://localhost:8050/#/experiments/2/runs/3d43ac8d8da24c49898b4a4476e97e73
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2      P2.5-P97.5   cobertura nuevos = 92.2 %  ancho = $221,271


🏃 View run intervalo_P10-P90_v2_geo at: http://localhost:8050/#/experiments/2/runs/3848712c6be54be4b2f77d487053c5c7
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2_geo  P10-P90      cobertura nuevos = 75.2 %  ancho = $104,012


🏃 View run intervalo_P7.5-P92.5_v2_geo at: http://localhost:8050/#/experiments/2/runs/c5abdf4621ab46b8aab5c3c1219d2816
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2_geo  P7.5-P92.5   cobertura nuevos = 80.5 %  ancho = $120,268


🏃 View run intervalo_P5-P95_v2_geo at: http://localhost:8050/#/experiments/2/runs/e8f0f3c3463c42beaa8f44a75102ecf6
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2_geo  P5-P95       cobertura nuevos = 85.0 %  ancho = $145,101


🏃 View run intervalo_P2.5-P97.5_v2_geo at: http://localhost:8050/#/experiments/2/runs/2a586260b84940b79eb3808487526897
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2_geo  P2.5-P97.5   cobertura nuevos = 91.1 %  ancho = $195,012


## 2. Un hiperparámetro a la vez (percentiles P7,5–P92,5)

Con el par de percentiles que apunta al 80 %, se prueban **4 valores de cada hiperparámetro** de los modelos cuantílicos, dejando los demás en la base.

| Hiperparámetro | Base | Valores probados |
|---|---|---|
| `n_estimators` | 600 | 200, 400, 1000, 1500 |
| `learning_rate` | 0,03 | 0,01, 0,02, 0,05, 0,1 |
| `num_leaves` | 31 | 15, 63, 127, 255 |
| `min_child_samples` | 30 | 5, 10, 50, 100 |

In [3]:
A_INF, A_SUP = 0.075, 0.925

pruebas = {
    "n_estimators": [200, 400, 1000, 1500],
    "learning_rate": [0.01, 0.02, 0.05, 0.1],
    "num_leaves": [15, 63, 127, 255],
    "min_child_samples": [5, 10, 50, 100],
}

for version in ["v2", "v2_geo"]:
    for parametro, valores in pruebas.items():
        for valor in valores:
            nombre = f"intervalo_{parametro}{valor}_{version}"
            m = intervalo(nombre, version, A_INF, A_SUP, **{parametro: valor})
            resultados.append({"corrida": nombre, "conjunto": version, "parametro": parametro,
                               "valor": valor, **m})
        print(f"{version:7s} {parametro} listo")

🏃 View run intervalo_n_estimators200_v2 at: http://localhost:8050/#/experiments/2/runs/529bc459cccd4320b79495b10a7cdc80
🧪 View experiment at: http://localhost:8050/#/experiments/2


🏃 View run intervalo_n_estimators400_v2 at: http://localhost:8050/#/experiments/2/runs/64867522e1a3402196535dbfa1856853
🧪 View experiment at: http://localhost:8050/#/experiments/2


🏃 View run intervalo_n_estimators1000_v2 at: http://localhost:8050/#/experiments/2/runs/3a71d4cd32074b588d078bd157e48a56
🧪 View experiment at: http://localhost:8050/#/experiments/2


🏃 View run intervalo_n_estimators1500_v2 at: http://localhost:8050/#/experiments/2/runs/281a4df1eaef4b5c9047a881417109c7
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2      n_estimators listo


🏃 View run intervalo_learning_rate0.01_v2 at: http://localhost:8050/#/experiments/2/runs/871f072299db441a95252c9db5c5fee3
🧪 View experiment at: http://localhost:8050/#/experiments/2


🏃 View run intervalo_learning_rate0.02_v2 at: http://localhost:8050/#/experiments/2/runs/0b17e3b0d7cc41729cc3d97462ae0640
🧪 View experiment at: http://localhost:8050/#/experiments/2


... (22 outputs left out)

🏃 View run intervalo_min_child_samples5_v2_geo at: http://localhost:8050/#/experiments/2/runs/92aad5f2eb4a40a9840a2b0c9ef06c7d
🧪 View experiment at: http://localhost:8050/#/experiments/2


🏃 View run intervalo_min_child_samples10_v2_geo at: http://localhost:8050/#/experiments/2/runs/bbd48cc43b45475a93e2602b1279f63e
🧪 View experiment at: http://localhost:8050/#/experiments/2


🏃 View run intervalo_min_child_samples50_v2_geo at: http://localhost:8050/#/experiments/2/runs/039c9f5bfcd840e0999ae972c1f68956
🧪 View experiment at: http://localhost:8050/#/experiments/2


🏃 View run intervalo_min_child_samples100_v2_geo at: http://localhost:8050/#/experiments/2/runs/28ea1e8faf1a45108b55d896510f3018
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2_geo  min_child_samples listo


## Resumen

Cobertura en anuncios nuevos y ancho del rango para cada prueba.

In [4]:
tabla = pd.DataFrame(resultados)
tabla.pivot_table(index=["parametro", "valor"], columns="conjunto",
                  values=["cobertura_nuevos", "ancho_mediano_cop"]).round(1)

ancho_mediano_cop           cobertura_nuevos  \
conjunto                                    v2    v2_geo               v2   
parametro         valor                                                     
learning_rate     0.01                150507.4  136061.2             84.6   
                  0.02                143270.0  125795.9             82.7   
                  0.05                134296.9  115169.5             80.5   
                  0.1                 127912.3  107257.0             79.5   
min_child_samples 5                   141359.0  123377.1             82.4   
                  10                  140373.8  121619.7             82.4   
                  50                  139828.7  122018.2             81.7   
                  100                 142447.1  123100.4             81.9   
n_estimators      200                 151082.7  135906.2             84.4   
                  400                 143557.6  125389.9             82.7   
                  1000                135704.6  113963.5             80.7   
                  1500                131858.4  110007.3             79.5   
num_leaves        15                  147165.4  129770.8             83.5   
                  63                  130150.8  110466.2             79.2   
                  127                 119719.3   99433.5             76.7   
                  255                 106141.6   86713.7             70.4   
percentiles       P10-P90             121063.4  104011.6             77.3   
                  P2.5-P97.5          221270.8  195012.0             92.2   
                  P5-P95              166381.8  145100.9             87.4   
                  P7.5-P92.5          139982.6  120268.1             81.5   

                                     
conjunto                     v2_geo  
parametro         valor              
learning_rate     0.01         84.1  
                  0.02         81.6  
                  0.05         78.4  
                  0.1          75.4  
min_child_samples 5            80.2  
                  10           80.1  
                  50           80.4  
                  100          80.1  
n_estimators      200          83.9  
                  400          81.6  
                  1000         78.4  
                  1500         76.9  
num_leaves        15           82.8  
                  63           76.6  
                  127          73.2  
                  255          66.6  
percentiles       P10-P90      75.2  
                  P2.5-P97.5   91.1  
                  P5-P95       85.0  
                  P7.5-P92.5   80.5

**Mejores configuraciones:** las más cercanas al 80 % de cobertura en anuncios nuevos y, entre ellas, las de rango más angosto.

In [5]:
columnas = ["conjunto", "parametro", "valor", "cobertura_nuevos", "cobertura",
            "ancho_mediano_cop", "razon_sup_inf", "desvio_meta"]
(tabla[tabla["desvio_meta"] <= 2]
 .sort_values(["conjunto", "ancho_mediano_cop"])
 .groupby("conjunto").head(5)[columnas].round(2))

,conjunto,parametro,valor,cobertura_nuevos,cobertura,ancho_mediano_cop,razon_sup_inf,desvio_meta
15,v2,learning_rate,0.1,79.53,79.90,127912.26,2.18,0.47
17,v2,num_leaves,63,79.23,80.26,130150.77,2.20,0.77
11,v2,n_estimators,1500,79.47,80.41,131858.43,2.23,0.53
14,v2,learning_rate,0.05,80.45,80.87,134296.95,2.27,0.45
10,v2,n_estimators,1000,80.70,81.07,135704.57,2.28,0.70
26,v2_geo,n_estimators,1000,78.43,79.82,113963.46,2.03,1.57
30,v2_geo,learning_rate,0.05,78.43,79.74,115169.53,2.03,1.57
5,v2_geo,percentiles,P7.5-P92.5,80.51,81.19,120268.12,2.12,0.51
37,v2_geo,min_child_samples,10,80.15,81.41,121619.74,2.15,0.15
38,v2_geo,min_child_samples,50,80.39,81.31,122018.19,2.12,0.39
